In [34]:
from tenerife_rag.config import LLMConfig

config = LLMConfig.from_env()
client = config.client()

config.public_dict()

response = client.responses.create(model=config.generation_model, input="Hello, world!", temperature=config.temperature, top_p=config.top_p)
print(response.output_text)

Hello! How can I assist you today?


In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from tenerife_rag.chunking import Chunk

pages = PyMuPDFLoader("../data/TENERIFE.pdf").load()
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
windows = splitter.split_documents(pages)

chunks = [
    Chunk(id=str(i), text=w.page_content, pages=[w.metadata["page"] + 1])
    for i, w in enumerate(windows)
]

print(len(chunks))
print(chunks[0].text)
print("-----------------------------------------------")
print(chunks[32].text, chunks[32].citation)

ImportError: cannot import name 'PyMuPDFLoader' from 'langchain_core.document_loaders' (d:\repos\tenerife_rag\.venv\Lib\site-packages\langchain_core\document_loaders\__init__.py)

In [36]:
from tenerife_rag.store import build_store, search

store = build_store(chunks, config.embeddings())

In [37]:
search(store, "playa de arena negra")

[(Chunk(id='8', text='o Playa de Benijo [vídeo - ubicación] \nPlaya de arena negra, si el día está despejado, de los atardeceres más \nespectaculares que podéis ver en la isla. \nPara llegar a esta playa, aparcar por aquí y, para acceder a la playa, se accede \ndesde aquí (puede ser que os encontréis con el acceso cortado con una valla, \npero da igual, podéis acceder igualmente). \n• La Orotava \nEste es el pueblo más bonito y maravilloso del mundo (no es porque yo sea de allí).', kind='ventana', path=[], pages=[8]),
  0.5812820818459936),
 (Chunk(id='25', text='o Playa del Camisón \nEsta es una playa pequeñita que a nosotros en concreto nos gusta mucho.', kind='ventana', path=[], pages=[20]),
  0.5109466661798824),
 (Chunk(id='29', text='• El Médano / La Tejita \nPlayas tipiquísimas de Tenerife y otra de las estampas de la isla con Montaña Roja al \nfondo. Nota: La Tejita es playa nudista para que lo tengáis en cuenta.  \n \n \n• Los Cristianos \nDe la zona de Los Cristianos que me v

In [38]:
search(store, "hay carta en un guachinche")

[(Chunk(id='31', text='atravesaréis una especie de camino de tierra, meteos sin miedo. Si \nvais a Tenerife y no probáis una experiencia en Guachinche, no habéis \nido a Tenerife. Os podría recomendar toda la carta, prácticamente – \nde hecho, no. No podría, ya que en un guachinche no hay carta. Y de \nbebidas solo hay tres cosas: vino, seven up para mezclar con el vino, y \nagua. That’s all. Mi recomendación realmente es que preguntéis a los \ncamareros y, si es vuestra primera experiencia en un guachinche, \nhacédselo saber y que os recomienden ellos para que os vayáis de 10) \n[Tripadvisor] (os podría recomendar más guachinches, pero diría que \neste es mi favorito) \n \nNota: los Guachinches son típicos en la zona norte de Tenerife. \nActualmente existen muchos restaurantes que se llaman Guachinches', kind='ventana', path=[], pages=[24]),
  0.46639605198608325),
 (Chunk(id='32', text='Nota: los Guachinches son típicos en la zona norte de Tenerife. \nActualmente existen muchos resta

In [39]:
INSTRUCTIONS: str = """Eres un guía turístico de Tenerife. Responde en español usando SOLO los fragmentos.
Cita cada dato con el id del fragmento entre corchetes, por ejemplo [S1].
Si los fragmentos no contienen la respuesta, dilo."""

def answer(question:str, top_k: int=3):
    hits = search(store, question, top_k)
    context = "\n".join(
        f"[S{i}] {chunk.citation} {chunk.text}" for i, (chunk, _) in enumerate(hits, 1)
    )
    
    response = client.responses.create(
        model=config.generation_model,
        instructions=INSTRUCTIONS,
        input=f"Fragmentos:\n{context}\n\nPregunta: {question}\n\nRespuesta:",
        temperature=config.temperature,
        top_p=config.top_p,
        max_output_tokens=config.max_output_tokens
    )
    
    return response.output_text, hits

In [40]:
answer("¿Qué playas de arena negra hay en Tenerife?", top_k=3)

('En Tenerife hay playas de arena negra como la Playa de Benijo, que además ofrece atardeceres espectaculares si el día está despejado [S2].',
 [(Chunk(id='29', text='• El Médano / La Tejita \nPlayas tipiquísimas de Tenerife y otra de las estampas de la isla con Montaña Roja al \nfondo. Nota: La Tejita es playa nudista para que lo tengáis en cuenta.  \n \n \n• Los Cristianos \nDe la zona de Los Cristianos que me venga a la mente os recomendaría la Playa de \nLos Cristianos o la Playa de Las Vistas y creo que poco más (la imagen es la Playa de \nLas Vistas).', kind='ventana', path=[], pages=[23]),
   0.6458496728143637),
  (Chunk(id='8', text='o Playa de Benijo [vídeo - ubicación] \nPlaya de arena negra, si el día está despejado, de los atardeceres más \nespectaculares que podéis ver en la isla. \nPara llegar a esta playa, aparcar por aquí y, para acceder a la playa, se accede \ndesde aquí (puede ser que os encontréis con el acceso cortado con una valla, \npero da igual, podéis acceder 

In [43]:
from datetime import timedelta

import requests

from tenerife_rag.weather import WeatherService, canary_today

weather = WeatherService()
today = canary_today()


def in_days(n: int) -> str:
    return (today + timedelta(days=n)).isoformat()

In [44]:
for days, place in [(1, "santa_cruz"), (2, "teide"), (3, "masca")]:
    print(weather.get_weather(in_days(days), place))

{'ok': True, 'fuente': 'Open-Meteo', 'modo': 'real', 'fecha': '2026-10-12', 'lugar_id': 'santa_cruz', 'lugar': 'Santa Cruz de Tenerife', 'estado': 'parcialmente nublado', 'weather_code': 2, 'temp_max_c': 25.8, 'temp_min_c': 20.6, 'prob_precipitacion_pct': 0, 'precipitacion_mm': 0.0, 'viento_max_kmh': 16.6}
{'ok': True, 'fuente': 'Open-Meteo', 'modo': 'real', 'fecha': '2026-10-13', 'lugar_id': 'teide', 'lugar': 'Parque Nacional del Teide (Las Cañadas)', 'estado': 'cubierto', 'weather_code': 3, 'temp_max_c': 14.6, 'temp_min_c': 10.6, 'prob_precipitacion_pct': 0, 'precipitacion_mm': 0.0, 'viento_max_kmh': 13.5}
{'ok': True, 'fuente': 'Open-Meteo', 'modo': 'real', 'fecha': '2026-10-14', 'lugar_id': 'masca', 'lugar': 'Masca', 'estado': 'cubierto', 'weather_code': 3, 'temp_max_c': 24.5, 'temp_min_c': 19.1, 'prob_precipitacion_pct': 0, 'precipitacion_mm': 0.0, 'viento_max_kmh': 14.0}


In [45]:
print(weather.get_weather(in_days(-1), "santa_cruz"))
print(weather.get_weather(in_days(30), "santa_cruz"))

def timeout_get(*args, **kwargs):
    raise requests.Timeout("simulated timeout")


offline = WeatherService(http_get=timeout_get)
print(offline.get_weather(in_days(1), "santa_cruz"))

Intento 1 de Open-Meteo fallido: Open-Meteo no respondió a tiempo.
Intento 2 de Open-Meteo fallido: Open-Meteo no respondió a tiempo.


{'ok': False, 'error': 'fecha_pasada', 'message': '2026-10-10 ya ha pasado; solo hay predicciones desde hoy.', 'fecha': '2026-10-10'}
{'ok': False, 'error': 'fuera_de_horizonte', 'message': 'Solo hay predicción hasta 2026-10-26 (unos 15 días vista).', 'fecha': '2026-11-10'}
{'ok': False, 'error': 'timeout', 'message': 'Open-Meteo no respondió a tiempo.'}


In [ ]:
from collections.abc import Callable
from dataclasses import dataclass

from pydantic import BaseModel

from tenerife_rag.weather import PLACES, WeatherArgs, WeatherService

@dataclass
class ToolSpec:
    schema: dict
    function: Callable[..., dict]
    args_model: type[BaseModel]

WEATHER_SCHEMA = {
    "type": "function",
    "name": "get_weather",
    "description": "Predicción del tiempo en Tenerife para una fecha (AAAA-MM-DD) y un lugar.",
    "strict": True,
    "parameters": {
        "type": "object",
        "properties": {
            "fecha": {"type": "string", "description": "Fecha en formato AAAA-MM-DD."},
            "lugar": {"type": ["string", "null"], "enum": [*PLACES, None], "description": "Zona de la isla."},
        },
        "required": ["fecha", "lugar"],
        "additionalProperties": False,
    },
}

weather_service = WeatherService()
TOOLS = {"get_weather": ToolSpec(WEATHER_SCHEMA, weather_service.get_weather, WeatherArgs)}


In [ ]:
from pydantic import ValidationError

from tenerife_rag.tools import log_tool_call, setup_logging

setup_logging()

def execute_tool_call(call, registry: dict[str, ToolSpec]) -> dict:
    spec = registry.get(call.name)
    try:
        if spec is None:
            result = {"ok": False, "error": "herramienta_desconocida", "message": call.name}
        else:
            args = spec.args_model.model_validate_json(call.arguments)
            result = spec.function(**args.model_dump())
    except ValidationError as exc:
        result = {"ok": False, "error": "argumentos_invalidos", "message": str(exc)}
    except Exception as exc:
        result = {"ok": False, "error": "error_interno", "message": type(exc).__name__}
    log_tool_call(call.name, call.arguments, result)
    return result


In [ ]:
import json

def ask_with_tools(question: str) -> str:
    conversation = [{"role": "user", "content": question_with_context(question)}]
    for _ in range(config.max_turns):
        response = client.responses.create(
            model=config.generation_model,
            instructions=INSTRUCTIONS,
            input=conversation,
            tools=[spec.schema for spec in TOOLS.values()],
            temperature=config.temperature,
            top_p=config.top_p,
            max_output_tokens=config.max_output_tokens,
        )
        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:
            return response.output_text
        conversation += response.output
        for call in calls:
            result = execute_tool_call(call, TOOLS)
            conversation.append(
                {"type": "function_call_output", "call_id": call.call_id, "output": json.dumps(result, ensure_ascii=False)}
            )
    raise RuntimeError(f"Sin respuesta tras {config.max_turns} llamadas al modelo")


In [ ]:
def instructions_for(today: date) -> str:
    return INSTRUCTIONS + f"\nHoy es {DIAS[today.weekday()]} {today.isoformat()}. Convierte fechas relativas a AAAA-MM-DD."